In [11]:
import pandas as pd

df = pd.read_csv("dataset.csv",encoding="cp949")

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11844\1905399877.py:3: DtypeWarning: Columns (24,33,40,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("dataset.csv",encoding="cp949")


In [ ]:
import pandas as pd

## 부실 라벨링 

# 1. 날짜 형식 변환 (이미 되어 있다면 생략 가능)
df['상장폐지일'] = pd.to_datetime(df['상장폐지일'], errors='coerce')

# 2. 부실 여부 컬럼 초기화 (기본값 0)
df['부실여부'] = 0

# 3. 상장폐지 전년도(t-1) 데이터에 1 부여
# 조건: 상장폐지일이 있고, (결산연도 == 상장폐지연도 - 1) 인 경우
condition_distress = (df['상장폐지일'].notnull()) & \
                     (df['연도'] == df['상장폐지일'].dt.year - 1)

df.loc[condition_distress, '부실여부'] = 1

# 4. 결과 확인 (부실 기업 수와 전체 비율)
print(df['부실여부'].value_counts())

부실여부
0    28329
1      161
Name: count, dtype: int64


In [2]:
import pandas as pd

# 1. 날짜 형식 변환 (상장폐지일이 문자열일 경우 대비)
df['상장폐지일'] = pd.to_datetime(df['상장폐지일'], errors='coerce')

# 2. '상장폐지연도' 추출 및 '결산연도'와 비교
# 상장폐지일이 있고(notnull), 결산연도(연도)가 상장폐지연도보다 크거나 같은 조건
condition = (df['상장폐지일'].notnull()) & (df['연도'] >= df['상장폐지일'].dt.year)

# 3. 해당 조건에 맞는 데이터 개수 확인
target_count = df[condition].shape[0]

print(f"상장폐지 연도 이후(포함) 데이터 개수: {target_count}개")

# (참고) 해당 데이터만 따로 보고 싶다면
# df_delisted_after = df[condition]

상장폐지 연도 이후(포함) 데이터 개수: 940개


In [3]:
df_delisted_after = df[condition]

df_delisted_after[['종목명','연도','상장폐지일']]

,종목명,연도,상장폐지일
1,에이스하이텍,2015,2015-07-10
2,에이스하이텍,2016,2015-07-10
3,에이스하이텍,2017,2015-07-10
4,에이스하이텍,2018,2015-07-10
5,에이스하이텍,2019,2015-07-10
...,...,...,...
23481,파티게임즈,2021,2020-09-09
23482,파티게임즈,2022,2020-09-09
23483,파티게임즈,2023,2020-09-09
23484,파티게임즈,2024,2020-09-09


In [4]:
import pandas as pd

# 1. 날짜 데이터 형식 변환 (에러 방지용)
df['상장일'] = pd.to_datetime(df['상장일'], errors='coerce')
df['상장폐지일'] = pd.to_datetime(df['상장폐지일'], errors='coerce')

# 2. 조건 설정
# 조건 1: 상장일의 연도가 2014년임
# 조건 2: 상장폐지일이 비어있음 (isnull)
condition = (df['상장일'].dt.year == 2014) & (df['상장폐지일'].isnull())

# 3. 데이터 선택 및 확인
normal_2014_firms = df[condition]

print(f"2014년 상장 및 현재 생존 기업 수: {normal_2014_firms['종목명'].nunique()}개")
print(f"해당 데이터 행 총 개수: {len(normal_2014_firms)}개")

# 결과 확인 (상단 5개)
print(normal_2014_firms[['종목명', '상장일', '상장폐지일']].head())

2014년 상장 및 현재 생존 기업 수: 63개
해당 데이터 행 총 개수: 693개
      종목명        상장일 상장폐지일
462  국일신동 2014-12-29   NaT
463  국일신동 2014-12-29   NaT
464  국일신동 2014-12-29   NaT
465  국일신동 2014-12-29   NaT
466  국일신동 2014-12-29   NaT


In [5]:
# 1. 상장일 데이터를 날짜 형식으로 변환
df['상장일'] = pd.to_datetime(df['상장일'], errors='coerce')

# 2. '상장 연도'보다 '결산 연도'가 작은 행들만 필터링 (삭제할 행 찾기)
# 예: 상장일이 2017년인데 연도가 2014, 2015, 2016인 경우
drop_condition = df['연도'] < df['상장일'].dt.year

# 3. 삭제 대상이 아닌(상장 이후인) 데이터만 남기기
df_cleaned = df[~drop_condition]

print(f"상장 전 데이터 {len(df) - len(df_cleaned)}행이 삭제되었습니다.")

상장 전 데이터 6131행이 삭제되었습니다.


In [6]:
import pandas as pd

df = pd.read_csv("dataset.csv",encoding="cp949")

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_11844\1905399877.py:3: DtypeWarning: Columns (24,33,40,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("dataset.csv",encoding="cp949")


In [13]:
import pandas as pd

# 1. 날짜 데이터 형식 변환 (연도 추출을 위해 필수)
df['상장일'] = pd.to_datetime(df['상장일'], errors='coerce')
df['상장폐지일'] = pd.to_datetime(df['상장폐지일'], errors='coerce')

# 2. 삭제 조건 설정 (합집합 방식)
# 조건 A: 결산연도가 상장연도보다 앞서는 경우 (상장 전)
cond_before_listing = df['연도'] < df['상장일'].dt.year

# 조건 B: 상장폐지일이 있고, 결산연도가 상장폐지연도와 같거나 큰 경우 (상폐 이후)
# (질문하신 논리에 따라 상폐 당해연도부터 삭제하도록 설정)
cond_after_delisting = (df['상장폐지일'].notnull()) & (df['연도'] >= df['상장폐지일'].dt.year)

# 3. 데이터 필터링 (두 조건 중 하나라도 해당되면 삭제 대상)
# ~ 기호는 '반전(Not)'을 의미하므로, 두 조건에 해당하지 않는 '유지' 데이터만 남깁니다.
df_final = df[~(cond_before_listing | cond_after_delisting)].copy()

# 4. 결과 확인
print(f"원래 데이터 행 수: {len(df)}")
print(f"정제 후 데이터 행 수: {len(df_final)}")
print(f"삭제된 총 행 수: {len(df) - len(df_final)}")

원래 데이터 행 수: 28490
정제 후 데이터 행 수: 21419
삭제된 총 행 수: 7071


In [8]:
import pandas as pd

# 모든 열 다 보기
pd.set_option('display.max_columns', None)

# 모든 행 다 보기 (데이터가 수만 건이면 렉이 걸릴 수 있으니 주의!)
pd.set_option('display.max_rows', None)

# 열 너비 제한 없이 다 보기 (내용이 길 때 유용)
pd.set_option('display.max_colwidth', None)


df_final.isnull().sum()

업체코드                     0
종목코드                     0
종목명                      0
업체명                      0
법인번호                     4
사업자번호                    4
설립일                      0
상장일                      0
상장폐지일                20364
시장구분                     0
결산월                      0
기업규모                     0
주식업종                  6405
소속기업집단                3528
NICS산업분류                23
표준산업대분류                  0
표준산업중분류                  0
연도                       0
01_총자본증가율              357
02_영업이익증가율             362
03_순이익증가율              359
04_자기자본증가율             576
05_매출액증가율              459
06_종업원수증가율            1001
07_매출액총이익율             345
08_매출액영업이익율            345
09_매출액순이익율             345
10_총자산영업이익율            150
11_총자산순이익율             148
12_자기자본영업이익율           150
13_자기자본순이익율            440
14_금융비용부담율             345
15_수지비율                276
16_사내유보대자기자본비율         277
17_총자본회전율              275
18_자기자본회전율             479
19_타인자본회전율             225
2

In [ ]:
df_final.to_excel("data_존속기업필터링.xlsx")

In [9]:
df.isnull().sum()

업체코드                     0
종목코드                     0
종목명                      0
업체명                      0
법인번호                    11
사업자번호                   11
설립일                      0
상장일                      0
상장폐지일                26433
시장구분                     0
결산월                      0
기업규모                     0
주식업종                  9515
소속기업집단                6424
NICS산업분류               231
표준산업대분류                  0
표준산업중분류                  0
연도                       0
01_총자본증가율             7019
02_영업이익증가율            7024
03_순이익증가율             7021
04_자기자본증가율            7276
05_매출액증가율             7125
06_종업원수증가율            7681
07_매출액총이익율            2389
08_매출액영업이익율           2389
09_매출액순이익율            2389
10_총자산영업이익율           6806
11_총자산순이익율            6804
12_자기자본영업이익율          6806
13_자기자본순이익율           3036
14_금융비용부담율            2389
15_수지비율               2170
16_사내유보대자기자본비율        2147
17_총자본회전율             2161
18_자기자본회전율            7178
19_타인자본회전율            6886
2

In [14]:
# 1. 정제 전 결측치 합계
null_before = df.isnull().sum()

# 2. 정제 후 결측치 합계
null_after = df_final.isnull().sum()

# 3. 두 결과를 하나로 합치기 (비교용 테이블)
compare_nulls = pd.DataFrame({
    '정제 전 (Full)': null_before,
    '정제 후 (Cleaned)': null_after,
    '줄어든 결측치 수': null_before - null_after
})

# 모든 행을 출력하도록 설정 (컬럼이 50개가 넘으므로)
import pandas as pd
pd.set_option('display.max_rows', None)

print("### 데이터 정제 전후 결측치 비교 ###")
print(compare_nulls)

### 데이터 정제 전후 결측치 비교 ###
                   정제 전 (Full)  정제 후 (Cleaned)  줄어든 결측치 수
업체코드                         0               0          0
종목코드                         0               0          0
종목명                          0               0          0
업체명                          0               0          0
법인번호                        11               4          7
사업자번호                       11               4          7
설립일                          0               0          0
상장일                          0               0          0
상장폐지일                    26433           20364       6069
시장구분                         0               0          0
결산월                          0               0          0
기업규모                         0               0          0
주식업종                      9515            6405       3110
소속기업집단                    6424            3528       2896
NICS산업분류                   231              23        208
표준산업대분류                      0               0 

In [16]:
# 부실여부(0과 1)의 개수를 각각 세어줍니다.
print(df_final['부실여부'].value_counts())

# 비율(%)로 보고 싶다면 이렇게 하세요.
print(df_final['부실여부'].value_counts(normalize=True) * 100)

부실여부
0    21258
1      161
Name: count, dtype: int64
부실여부
0    99.248331
1     0.751669
Name: proportion, dtype: float64


In [17]:
# 연도별로 부실여부(0, 1)의 합계와 전체 개수를 계산
yearly_stats = df_final.groupby('연도')['부실여부'].agg(['count', 'sum'])

# 컬럼명 변경 (이해하기 쉽게)
yearly_stats.columns = ['전체 데이터(N)', '부실 데이터(Y)']

# 연도별 부실률(%) 계산
yearly_stats['부실률(%)'] = (yearly_stats['부실 데이터(Y)'] / yearly_stats['전체 데이터(N)']) * 100

# 결과 출력
print(yearly_stats)

      전체 데이터(N)  부실 데이터(Y)    부실률(%)
연도                                  
2014       1566         18  1.149425
2015       1653         12  0.725953
2016       1714         15  0.875146
2017       1781         14  0.786075
2018       1864          4  0.214592
2019       1955         17  0.869565
2020       2020         20  0.990099
2021       2100         19  0.904762
2022       2162          9  0.416281
2023       2261         16  0.707651
2024       2343         17  0.725566


In [ ]:
df_final.to_csv("기업존속케이스제거후데이터셋.csv", index=False, encoding="cp949")